In [2]:
using ModelingToolkit
using ModelingToolkit: t_nounits as t, D_nounits as D
using OrdinaryDiffEq
using Plots

In [5]:
#Time-saving naming scheme
const NAME_DICT = Dict{DataType, Int}()
function get_unique_name(obj)
    type = typeof(obj)
    if !haskey(NAME_DICT, type)
        NAME_DICT[type] = 0
    end
    NAME_DICT[type] +=1
    return Symbol(lowercase(string(nameof(T))), "_", NAME_DICT[type])
end
#Base ion channel class; will be replicated for all base-class-able components
@component function BaseIonChannel(;name, v_in, conductance, reversal_potential, kwargs...)
    pars = @parameters begin
        g = conductance
        E = reversal_potential
    end

    vars = @variables begin
        i(t)
        m(t) = 0.0
        h(t) = 0.0
    end
    eqs = []

    return ODESystem(eqs, t, vars, pars; name=name)
end


BaseIonChannel (generic function with 1 method)

In [ ]:
eNa = 50.0      # Sodium
eK = -80.0      # Potassium
eh = -20.0      # H-current
eleak = -50.0   # Leak current

# Conductances
gNabar = 100.0   # Sodium
gCaSbar = 3.0    # Slow calcium
gCaTbar = 1.3    # Transient calcium
gKabar = 5.0     # A-type potassium
gKCabar = 10.0   # Calcium-activated potassium
gKdrbar = 20.0   # Delayed rectifier potassium
ghbar = 0.5      # H-current
gleak = 0.01     # Leak

# Other Parameters
tauCa = 20.0     # Calcium time constant
Ca = 0.05
eCa = (500.0) * (8.6174e-5) * (283.15) * (log((3000.0 / Ca)))

20.0

In [ ]:
@component function CalciumDynamics(;name, v_in)
    pars = @parameters begin
        tau_Ca = 20.0  # Calcium time constant
        Ca_base = 0.05  # Base calcium level
    end
    
    vars = @variables begin
        Ca(t) = 0.05  # Calcium concentration
        E_Ca(t)       # Calcium reversal potential
    end
    
    # Get calcium currents from connected channels
    # This assumes calcium channels will connect their currents to this component
    @variables I_Ca(t)
    
    eqs = [
        # Calcium concentration dynamics
        D(Ca) ~ (1/tau_Ca) * (Ca_base + 0.94*(I_Ca) - Ca),
        
        # Dynamic calcium reversal potential using Nernst equation
        E_Ca ~ (500.0) * (8.6174e-5) * (283.15) * log((3000.0 / Ca))
    ]
    
    return ODESystem(eqs, t, vars, pars; name=name)
end

In [ ]:
#Define specific ion channels
@component function HHSodiumChannel(;name, v_in, conductance=gNabar, reversal_potential=eNa, kwargs...)
    @named base = IonChannel(v_in=v_in, conductance=conductance, reversal_potential=reversal_potential)

    m_inf(v) = 1.0 / (1.0 + exp((v + 25.5) / -5.29))
    h_inf(v) = 1.0 / (1.0 + exp((v + 48.9) / 5.18))
    tau_m(v) = 1.32 - 1.26 / (1 + exp((v + 120.0) / -25.0))
    tau_h(v) = (0.67 / (1.0 + exp((v + 62.9) / -10.0))) * (1.5 + 1.0 / (1.0 + exp((v + 34.9) / 3.6)))
    
    # Channel dynamics
    eqs = [
        D(base.m) ~ (m_inf(v_in) - base.m) / tau_m(v_in),
        D(base.h) ~ (h_inf(v_in) - base.h) / tau_h(v_in),
        base.i ~ base.g * base.m^3 * base.h * (base.E - v_in)
    ]
    
    return ODESystem(eqs, t, vars, []; systems=[base], name=name)
end

#CaS m starts at 20.0, not at 0.0 . OVERWRITE SOMEWHERE SOMEHOW SOMEWHY
@component function CaSChannel(;name, v_in, conductance=gCaSbar) 
    @named base = IonChannel(v_in=v_in, conductance=conductance, reversal_potential=CalciumDynamics)

    m_inf(v) = 1.0 / (1.0 + exp((v + 33.0) / -8.1))
    h_inf(v) = 1.0 / (1.0 + exp((v + 60.0) / 6.2))
    tau_m(v) = 1.4 - 7.0 / (1 + exp((v + 27.0) / -10.0) + exp((v + 70.0) / -13.0))
    tau_h(v) = 60.0 + 150.0 / (exp((V + 55.0) / 9.0) + exp((V + 65.0) / -16.0))

    eqs = [
        D(base.m) ~ (m_inf(v_in) - base.m) / tau_m(v_in),
        D(base.h) ~ (h_inf(v_in) - base.h) / tau_h(v_in),
        base.i ~ base.g * base.m^3 * base.h * (base.E - v_in)
    ]

    return ODESystem(eqs, t, vars, []; systems=[base], name=name)
end

@component function CaTChannel(;name, v_in, conductance=gCaTbar)
    @named base = IonChannel(v_in=v_in, conductance=conductance, reversal_potential=CalciumDynamics)

    m_inf(v) = 1.0 / (1.0 + exp((v + 27.1) / -7.2))
    h_inf(v) = 1.0 / (1.0 + exp((v + 32.1) / 5.5))
    tau_m(v) = 21.7 - 21.3 / (1.0 + exp((v + 68.1) / -20.5)) 
    tau_h(v) = 105.0 + 89.8 / (1.0 + exp((V + 55.0) / 16.9))

    eqs = [
        D(base.m) ~ (m_inf(v_in) - base.m) / tau_m(v_in),
        D(base.h) ~ (h_inf(v_in) - base.h) / tau_h(v_in),
        base.i ~ base.g * base.m^3 * base.h * (base.E - v_in)
    ]

    return ODESystem(eqs, t, vars, []; systems=[base], name=name)
end

@component function KaChannel(;name, v_in, conductance=gKabar, reversal_potential=eK)
    @named base = IonChannel(v_in=v_in, conductance=conductance, reversal_potential=reversal_potential)

    m_inf(v) = 1.0 / (1.0 + exp((v + 27.1) / -7.2))
    h_inf(v) = 1.0 / (1.0 + exp((v + 32.1) / 5.5))
    tau_m(v) = 21.7 - 21.3 / (1.0 + exp((v + 68.1) / -20.5)) 
    tau_h(v) = 105.0 + 89.8 / (1.0 + exp((V + 55.0) / 16.9))

    eqs = [
        D(base.m) ~ (m_inf(v_in) - base.m) / tau_m(v_in),
        D(base.h) ~ (h_inf(v_in) - base.h) / tau_h(v_in),
        base.i ~ base.g * base.m^3 * base.h * (base.E - v_in)
    ]

    return ODESystem(eqs, t, vars, []; systems=[base], name=name)
end

Base.Meta.ParseError: ParseError:
# Error @ c:\Users\eloya\source\thesis\neuronbuilder\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W1sZmlsZQ==.jl:83:37

@component function CalciumChannel()
#                                   └ ── premature end of input

In [ ]:
@component function HHNeuron(;name, input_current=0.0, kwargs...)
    pars = @parameters begin
        C = 0.01
        gap_C = 0.02
        spike_threshold=-40.0
    end

    vars = @variables begin
        v(t) = -70
        i(t)#, [connect=Flow]
    end
    systems = @named begin
        so = SodiumChannel(; v_in = v)
        po = PotassiumChannel(; v_in = v)
        #ca = CalciumChannel(; v_in = v)
        l = LeakChannel(; v_in = v)
    end
    
    # Sum the currents from ion channels first
    channel_currents = sum(el.i for el in systems)
    # Then add the input current
    eqs = [
        C * D(v) ~ so.i+po.i+l.i + input_current   
    ]
    return ODESystem(eqs, t, vars, pars; systems, name)
end